In [ ]:
from dotenv import load_dotenv
load_dotenv(r"..\..\.env")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
CHROMA_PATH = r"..\..\notebooks\chroma_db_langchain"  # separate folder from notebook 01's chroma_db/
COLLECTION_NAME = "ml_book_langchain"

EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-nano"

In [12]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vectorstore = Chroma(
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_PATH,
)


In [13]:
vectorstore._collection.count()

973

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

retriever.invoke("What is supervised learning?")

[Document(id='cf9c0a90-0810-4967-8299-57cc7a67e535', metadata={'total_pages': 392, 'source': 'E:\\Imarticus_ Learning\\Jobs - 2026\\Models\\data\\raw\\[ML] Introduction to Machine Learning with Python (2017).pdf', 'trapped': '/False', 'creationdate': '2016-09-21T13:04:39+00:00', 'page': 15, 'author': 'Andreas C. Müller and Sarah Guido', 'title': 'Introduction to Machine Learning with Python', 'moddate': '2016-09-22T10:41:19-04:00', 'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'page_label': '2', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)'}, page_content='learning algorithms because a “teacher” provides supervision to the algorithms in the\nform of the desired outputs for each example that they learn from. While creating a\ndataset of inputs and outputs is often a laborious manual process, supervised learning\nalgorithms are well understood and their performance is easy to measure. If your\napplication can be formulated as 

In [ ]:
prompt_template_str = """
  Your task is to explain the concept of {concept} to me in a way that is:

  1. Clear and intuitive
  2. Concise (in under 100 words)
  3. Tailored specifically to me and what I already know

  Base your explanation on the following excerpts from the reference book.
  If the excerpts do not cover the concept, say so instead of making something up.

  Excerpts:
  {context}

  Use the following information about me to personalize your explanations:

  - Background: Aspiring Data Scientist in the Automotive Industry
  - Professional Interests: Extracting structured information from engineering documents, data pipelines
  - Personal Interests: Automotive technology, AI applications in engineering

  The personalization should be subtle and natural. Avoid forced references to my background that don't genuinely enhance my understanding of the concept.
  """
  
  # Create a prompt template
prompt_template = ChatPromptTemplate.from_template(prompt_template_str)

# Create a model interface
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [25]:
def generate_explanation(concept):
    chain = (
    {"context": retriever | format_docs, "concept": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser())
    answer = chain.invoke(concept)
    return answer

In [26]:
generate_explanation('RAG')

'RAG (Retrieval-Augmented Generation) is an AI approach that combines data retrieval with language generation. It retrieves relevant information from large documents or databases—like engineering manuals—and feeds it into a language model to generate accurate, context-aware responses. For a data scientist in automotive tech, RAG enables precise extraction of structured info from complex documents, improving data pipelines and insights. By integrating retrieval with generation, RAG ensures that AI outputs are both knowledgeable and relevant, making it ideal for applications needing accurate, sourced information in engineering contexts.'

- copy the directory loaction
- open command prompt
- type cd copied location
- in anaconda prompt conda create -n env_name python = 3.12 -y
- conda activate <env_name>
- pip install the requirements.txt
- pip install the jupyter Notebook
- open VS code from the saved dicrectory
    - code .
- If the Vector DB not created already. create a VDB 
- Import all the nessary libs and api key
- define the chroma path where thr db needs to be saved
- define the collection name
- Define Embedding model
- using vectorstore import the VDB 
- 